# Scrapper for data of Given car (A90 (supra) etc)

In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
from playwright.sync_api import sync_playwright


In [ ]:

lap_time_url = "https://www.supramkv.com/threads/official-lap-times-thread.17634/"
power_guide_forum = "https://www.supramkv.com/threads/gr-supra-101-routes-to-big-power.17670/"
race_Track_forum = "https://www.supramkv.com/threads/race-tracking-your-supra-information-exchange.16389/"
car_dyno_forum = "https://www.supramkv.com/threads/car-driver-dyno-results-the-2020-toyota-supra-makes-more-power-than-toyota-claims.1814/"
exhaust_poll_forum = "https://www.supramkv.com/threads/best-exhaust-systems-in-the-market.4841/"
time_attack_build_forum = "https://www.supramkv.com/threads/tbks-time-attack-build-mk-brx.27329/"
street_build_forum = "https://www.supramkv.com/threads/mimosas-street-build.8686/"

# URls
urls_dict = {
    'lap_times': lap_time_url,
    'power_guide': power_guide_forum,
    'race_track': race_Track_forum,
    'car_dyno': car_dyno_forum,
    'exhaust_poll' : exhaust_poll_forum,
    'time_attack_build' : time_attack_build_forum,
    'street_build_fourm' : street_build_forum}


In [4]:
# Scrape all pages of a forum thread and extract post data, links, images, and YouTube videos (robust pagination handling, using next button class)
def scrape_forum_thread(thread_url, max_pages=100):
    import urllib.parse
    posts = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.96 Safari/537.36'
    }
    url = thread_url
    for page in range(1, max_pages+1):
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f'Failed to fetch page {page}: {response.status_code}')
            break
        soup = BeautifulSoup(response.text, 'html.parser')
        for post in soup.find_all('article', class_='message'):
            author = post.find('h4', class_='message-name')
            author = author.text.strip() if author else None
            content_div = post.find('div', class_='bbWrapper')
            if not content_div:
                content_div = post.find('div', class_='message-body')
            content = content_div.get_text(separator='\n', strip=True) if content_div else None
            date = post.find('time')
            date = date['datetime'] if date and date.has_attr('datetime') else None
            post_id = post.get('data-content') or post.get('id')
            # Extract links (excluding YouTube iframes, which are handled separately)
            links = []
            if content_div:
                for a in content_div.find_all('a', href=True):
                    links.append(urllib.parse.urljoin(thread_url, a['href']))
            images = []
            if content_div:
                for img in content_div.find_all("img"):
                    # candidate sources in priority order
                    candidates = []
                    # 1) lazy-load attribute often holds the real file
                    if img.get("data-src"):
                        candidates.append(img.get("data-src"))
                    # 2) normal src
                    if img.get("src"):
                        candidates.append(img.get("src"))

                    parent_a = img.find_parent("a", href=True)
                    if parent_a and parent_a.get("href"):
                        candidates.append(parent_a.get("href"))
                    # choose first usable candidate
                    chosen = None
                    for c in candidates:
                        if not c:
                            continue
                        c = c.strip()
                        # skip inline/base64 images
                        if c.startswith("data:"):
                            continue
                        # skip obvious smilies/emoji/icons (tune this if needed)
                        if "smilie" in c.lower() or "emoji" in c.lower():
                            continue
                        chosen = urllib.parse.urljoin(thread_url, c)
                        break

                    if chosen and chosen not in images:
                        images.append(chosen)
            #Capture any youtube videos (for later)
            youtube_videos = []
            if content_div:
                # 1️⃣ Handle iframe embeds
                for iframe in content_div.find_all("iframe", src=True):
                    src = iframe["src"]

                    if "youtube.com" in src or "youtu.be" in src:
                        # Extract video ID
                        video_id = None

                        # embed format
                        if "youtube.com/embed/" in src:
                            video_id = src.split("youtube.com/embed/")[1].split("?")[0]

                        # short link format
                        elif "youtu.be/" in src:
                            video_id = src.split("youtu.be/")[1].split("?")[0]

                        if video_id:
                            clean_url = f"https://www.youtube.com/watch?v={video_id}"
                            if clean_url not in youtube_videos:
                                youtube_videos.append(clean_url)

                # 2️⃣ Handle raw mediaembed spans (extra safe fallback)
                for span in content_div.find_all("span", attrs={"data-s9e-mediaembed": "youtube"}):
                    iframe = span.find("iframe", src=True)
                    if iframe:
                        src = iframe["src"]
                        if "youtube.com/embed/" in src:
                            video_id = src.split("youtube.com/embed/")[1].split("?")[0]
                            clean_url = f"https://www.youtube.com/watch?v={video_id}"
                            if clean_url not in youtube_videos:
                                youtube_videos.append(clean_url)
            posts.append({'author': author, 'content': content, 'date': date, 'post_id': post_id, 'links': links, 'images': images, 'youtube_videos': youtube_videos})
        # Find the next page link using the next button class and update url, else break
        next_link = soup.find('a', class_='pageNav-jump--next')
        if next_link and next_link.get('href'):
            url = urllib.parse.urljoin(thread_url, next_link['href'])
        else:
            break
    return posts

In [58]:
scraped_posts = scrape_forum_thread(power_guide_forum, max_pages=50)
df = pd.DataFrame(scraped_posts)
df

,author,content,date,post_id,links,images,youtube_videos
0,D3ad_Hand,Since the forums are ever more active and new ...,2023-04-29T02:16:05-0700,post-276037,"[https://www.supramkv.com/members/4250/, https...",[],[]
1,jtsang25,If this doesn't get pinned it's going to get l...,2023-04-29T03:22:23-0700,post-276041,[],[],[]
2,D3ad_Hand,jtsang25 said:\nIf this doesn't get pinned it'...,2023-04-29T03:36:03-0700,post-276042,[https://www.supramkv.com/goto/post?id=276041],[],[]
3,DSG Performance,Very thorough and comprehensive! Fantastic,2023-04-29T04:52:08-0700,post-276043,[],[],[]
4,Basura,"Bookmarked just in case, great info!",2023-04-29T04:57:08-0700,post-276044,[],[],[]
...,...,...,...,...,...,...,...
484,i3igpete,Bflood said:\nFlex kit he offers\nClick to exp...,2025-12-10T11:44:25-0800,post-526748,[https://www.supramkv.com/goto/post?id=526329],[],[]
485,Loco38SUP,i3igpete said:\nI don't know much about flex k...,2025-12-11T00:59:11-0800,post-526955,[https://www.supramkv.com/goto/post?id=526748],[],[]
486,SavagePassenger,Loco38SUP said:\nThis can be mitigated by wrap...,2025-12-11T15:57:38-0800,post-527152,[https://www.supramkv.com/goto/post?id=526955],[],[]
487,m340_B58,"If I upgrade my transmission (spool stage 2), ...",2026-02-11T09:05:49-0800,post-547187,[],[],[]


In [5]:
#loop through url dictionary and capture data and name csv "lap_times_csv" etc
all_data = {}
for key, url in urls_dict.items():
    posts = scrape_forum_thread(url, max_pages=300)
    all_data[key] = pd.DataFrame(posts)
    all_data[key].to_csv(f'{key}_csv.csv', index=False)
    print(f'Saved {key}_csv.csv with {len(posts)} posts')

Saved lap_times_csv.csv with 207 posts
Saved power_guide_csv.csv with 489 posts
Saved race_track_csv.csv with 4383 posts
Saved car_dyno_csv.csv with 172 posts
Saved exhaust_poll_csv.csv with 821 posts
Saved time_attack_build_csv.csv with 504 posts
Saved street_build_fourm_csv.csv with 589 posts


In [ ]:
import pickle

with open('data/car_data_pickle/gr_supra_all_data.pkl', 'wb') as f:
    pickle.dump(all_data, f)

In [55]:
race_track_df = all_data['race_track']

In [56]:
race_track_df

,author,content,date,post_id,links,images,youtube_videos
0,Todday1,All - I do not do a lot of these Forum things ...,2023-01-21T08:07:16-0800,post-252782,[],[],[]
1,FLtrackdays,Absolutely! There’s an old thread on this. Har...,2023-01-21T08:39:59-0800,post-252793,[https://www.supramkv.com/members/9528/],[],[]
2,Todday1,I do not know how to do the spread sheet thin...,2023-01-21T08:57:04-0800,post-252797,[],[],[]
3,RichSC,Fuel starve sucks and be nice to your transmis...,2023-01-21T09:51:37-0800,post-252806,[],[],[]
4,Matter,You instruct after 2 years? Impressive.\nHow ...,2023-01-21T15:28:50-0800,post-252857,[],[],[]
...,...,...,...,...,...,...,...
745,Afterfire,3TMagnetMan said:\nHow do you like the HKS Spr...,2023-06-07T06:46:59-0700,post-285028,[https://www.supramkv.com/goto/post?id=284859],[],[]
746,Marsone,"Slightly off topic, but anyone here going to b...",2023-06-07T09:23:11-0700,post-285070,[],[],[]
747,razorlab,Todday1 said:\nI do not have a data logger so ...,2023-06-07T10:19:23-0700,post-285081,[https://www.supramkv.com/goto/post?id=284861],[https://cdn.supramkv.com/attachments/96/96546...,[]
748,Traxion,"Marsone said:\nSlightly off topic, but anyone ...",2023-06-07T11:06:22-0700,post-285104,[https://www.supramkv.com/goto/post?id=285070],[],[]
